# SDXL Dataset Generation — Colab Runner

Runs `generate_dataset.py` against COCO val2017 to produce the 1200-sample SDXL manipulation benchmark.

**Before running:** Runtime → Change runtime type → GPU (T4 or better).

**Important:** generation writes to local Colab disk (`/content/data`), not directly to the Drive
mount. Checkpoints to Drive happen as a single tar archive per chunk (step 9), not as hundreds of
individual file writes — writing (or even reading) many small files against the Drive FUSE mount
saturates Drive's API quota, which is what broke the notebook's autosave and caused data loss on
the last disconnect.

## 1. Confirm a GPU is actually attached

In [ ]:
!nvidia-smi

import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU attached. Go to Runtime > Change runtime type > select a GPU, then re-run this cell."
    )
print("GPU check passed.")

## 2. Mount Drive and set PROJECT_DIR

Assumes the project (this repo: `generate_dataset.py`, `configs/`, `data/`) has been uploaded once to
`MyDrive/localization_project`. Edit `PROJECT_DIR` below if you used a different path. Drive here
persists the code, config, and periodic checkpoints of `data/` across runtime resets — it is not the
live write target during generation (see step 6).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/localization_project"

required = ["generate_dataset.py", "configs/generate_dataset_config.json"]
missing = [p for p in required if not (Path(PROJECT_DIR) / p).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing under {PROJECT_DIR}: {missing}. "
        "Upload the project to this Drive path (or fix PROJECT_DIR above) before continuing."
    )
print("PROJECT_DIR OK:", PROJECT_DIR)

## 3. Install dependencies

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors xformers tqdm

## 4. Verify the environment before touching the model

Fails fast with a clear message instead of a deep pipeline-load traceback.

In [ ]:
import torch
import diffusers

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("diffusers:", diffusers.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() is False — GPU runtime not active. Fix step 1 first.")

try:
    import xformers
    print("xformers:", xformers.__version__)
except ImportError:
    print("xformers not importable — will disable enable_xformers in the config below.")

## 5. Download COCO val2017 (idempotent — skips if already present)

Uses `unzip -n` (never overwrite) so re-running this cell never triggers an interactive
y/n/A/N/r prompt that would silently block the notebook. Already local (`/content/coco`) — this is a
read-only input, not generation output, so there's no Drive-sync risk here.

In [ ]:
import os

coco_dir = "/content/coco"
images_dir = os.path.join(coco_dir, "val2017")
ann_file = os.path.join(coco_dir, "annotations", "instances_val2017.json")

os.makedirs(coco_dir, exist_ok=True)

if not os.path.isdir(images_dir) or len(os.listdir(images_dir)) < 5000:
    !wget -q -O /content/val2017.zip http://images.cocodataset.org/zips/val2017.zip
    !unzip -n -q /content/val2017.zip -d {coco_dir}
else:
    print("val2017 images already present, skipping download.")

if not os.path.isfile(ann_file):
    !wget -q -O /content/annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -n -q /content/annotations_trainval2017.zip -d {coco_dir}
else:
    print("annotations already present, skipping download.")

assert os.path.isdir(images_dir) and len(os.listdir(images_dir)) >= 5000, "val2017 images missing/incomplete"
assert os.path.isfile(ann_file), "instances_val2017.json missing"
print("COCO ready:", images_dir, "|", ann_file)

## 6. Local working directory + resume from a single Drive checkpoint archive

This is the fix for the data-loss/autosave issue. `generate_dataset.py` writes an image copy, a mask
PNG, a metadata JSON, and rewrites `manifest.json` for *every sample* — hundreds of small writes.
Doing that straight against the Drive FUSE mount saturates Drive's sync, causing exactly what you saw:
the notebook's own autosave failing, and a runtime disconnect dropping whatever hadn't actually
finished syncing even though the script logged it as completed. Even reading many small files back
*from* Drive (e.g. via `rsync` against the mount) hits the same per-file latency wall and can take
many minutes for what should be a small amount of data.

Fix: never touch the Drive mount file-by-file. Generate to local disk (`/content/data`, fast and fully
durable within the session). To checkpoint, `tar` the whole directory into **one file** locally, then
copy that single file to Drive — one big sequential write instead of hundreds of small FUSE ops. To
resume after a disconnect, this cell does the reverse: copy the one checkpoint file down from Drive and
extract it locally.

In [ ]:
import os
import subprocess
import tarfile
from pathlib import Path

LOCAL_DATA_DIR = "/content/data"
LOCAL_CHECKPOINT = "/content/data_checkpoint.tar"
DRIVE_CHECKPOINT = f"{PROJECT_DIR}/data_checkpoint.tar"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if os.path.exists(DRIVE_CHECKPOINT):
    print(f"Found a Drive checkpoint at {DRIVE_CHECKPOINT}; restoring locally (single-file copy)...")
    subprocess.run(["cp", DRIVE_CHECKPOINT, LOCAL_CHECKPOINT], check=True)
    with tarfile.open(LOCAL_CHECKPOINT) as tf:
        tf.extractall("/content")
    print("Restored data/ from Drive checkpoint.")
else:
    print("No Drive checkpoint found yet; starting with an empty local data/ directory.")

def sync_to_drive() -> None:
    """Tar local data/ into one file and copy it to Drive. Call after each chunk, never mid-generation."""
    with tarfile.open(LOCAL_CHECKPOINT, "w") as tf:
        tf.add(LOCAL_DATA_DIR, arcname="data")
    tmp_drive_path = DRIVE_CHECKPOINT + ".tmp"
    subprocess.run(["cp", LOCAL_CHECKPOINT, tmp_drive_path], check=True)
    os.replace(tmp_drive_path, DRIVE_CHECKPOINT)

local_metadata_dir = Path(LOCAL_DATA_DIR, "metadata")
local_count = len(list(local_metadata_dir.glob("*.json"))) if local_metadata_dir.exists() else 0
print(f"Local metadata records after restore: {local_count}")

## 7. Update config for Colab

`device` must be `"cuda"`, not just the fp16/xformers/offload flags — those only take effect when
`device == "cuda"` in `generate_dataset.py`, otherwise generation silently falls back to CPU.

`data_dir` here is set to `LOCAL_DATA_DIR` for clarity, though every invocation below also passes
`--data-dir {LOCAL_DATA_DIR}` explicitly as an override.

In [ ]:
import json

config_path = f"{PROJECT_DIR}/configs/generate_dataset_config.json"
with open(config_path) as f:
    cfg = json.load(f)

cfg["data_dir"] = LOCAL_DATA_DIR
cfg["coco_images_dir"] = images_dir
cfg["coco_annotations_file"] = ann_file
cfg["device"] = "cuda"
cfg["torch_dtype"] = "float16"
cfg["enable_xformers"] = True
cfg["enable_cpu_offload"] = True
cfg["dry_run"] = False

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

for key in ("data_dir", "coco_images_dir", "coco_annotations_file", "device", "torch_dtype",
            "enable_xformers", "enable_cpu_offload", "dry_run"):
    print(f"{key}: {cfg[key]}")

## 8. Smoke test — 3 real samples

Confirms the pipeline actually loads and generates on GPU, writing to local disk, before committing to
the full 1200-image run.

In [ ]:
!cd {PROJECT_DIR} && python generate_dataset.py --config configs/generate_dataset_config.json --data-dir {LOCAL_DATA_DIR} --target-count 3 2>&1 | tee /content/smoke_test.log

with open("/content/smoke_test.log") as f:
    log = f.read()
if "Traceback" in log or "Fatal error" in log:
    raise RuntimeError("Smoke test failed — see /content/smoke_test.log above for the traceback.")
print("Smoke test passed.")

sync_to_drive()
print("Synced smoke-test checkpoint to Drive.")

## 9. Full run — 1200 samples, chunked with tar checkpoints to Drive

Runs `generate_dataset.py` in chunks of `CHUNK_SIZE`, tar-checkpointing local → Drive after each chunk.
The manifest/resume logic means each chunk only generates what's missing, so this is equivalent to one
1200-sample run but bounds disconnect data-loss to a single chunk (~200 samples ≈ 30-40 min) instead
of the whole run. If the runtime disconnects, just re-run cells 6 and 9 — cell 6 restores local from
Drive's last checkpoint archive and this cell picks up from there.

In [ ]:
CHUNK_SIZE = 200
TARGET = 1200
LOG_PATH = "/content/full_run.log"

chunk_targets = list(range(CHUNK_SIZE, TARGET, CHUNK_SIZE)) + [TARGET]

for chunk_target in chunk_targets:
    print(f"\n=== Generating up to {chunk_target}/{TARGET} ===")
    !cd {PROJECT_DIR} && python generate_dataset.py --config configs/generate_dataset_config.json --data-dir {LOCAL_DATA_DIR} --target-count {chunk_target} 2>&1 | tee -a {LOG_PATH}

    with open(LOG_PATH) as f:
        tail = f.read()[-2000:]
    if "Traceback" in tail or "Fatal error" in tail:
        sync_to_drive()
        raise RuntimeError(
            f"generate_dataset.py failed at target_count={chunk_target} — see {LOG_PATH}. "
            "Progress through the previous chunk is safely checkpointed to Drive."
        )

    sync_to_drive()
    print(f"Checkpoint synced to Drive at {chunk_target}/{TARGET}.")

print("Full run complete and fully synced to Drive.")

## 10. Verify results

In [ ]:
import json
import subprocess
import tarfile
from collections import Counter

with open(f"{LOCAL_DATA_DIR}/manifest.json") as f:
    local_manifest = json.load(f)
print("Local manifest status counts:", dict(Counter(e["status"] for e in local_manifest.values())))
print("Local manifest total:", len(local_manifest))

check_dir = "/content/drive_checkpoint_check"
subprocess.run(["rm", "-rf", check_dir], check=True)
with tarfile.open(DRIVE_CHECKPOINT) as tf:
    tf.extractall(check_dir)
with open(f"{check_dir}/data/manifest.json") as f:
    drive_manifest = json.load(f)
print("Drive checkpoint manifest total:", len(drive_manifest), "(should equal local if the last sync succeeded)")